In [4]:
# Step 1: Import the libraries we need

from collections import deque   # used for BFS queue
import heapq                    # used for A* priority queue
import time                     # used to measure runtime

In [8]:
# Step 2: Create a larger maze

# S = Start
# G = Goal
# . = Open path
# # = Wall/obstacle

maze = [
    ['S', '.', '.', '.', '#', '.', '.', '.'],
    ['#', '#', '#', '.', '#', '.', '#', '.'],
    ['.', '.', '.', '.', '.', '.', '#', '.'],
    ['.', '#', '#', '#', '#', '.', '#', '.'],
    ['.', '.', '.', '.', '#', '.', '.', '.'],
    ['#', '#', '#', '.', '#', '#', '#', '.'],
    ['.', '.', '.', '.', '.', '.', '.', '.'],
    ['.', '#', '#', '#', '#', '#', '#', 'G']
]

rows = len(maze)
cols = len(maze[0])

for row in maze:
    print(row)
    

['S', '.', '.', '.', '#', '.', '.', '.']
['#', '#', '#', '.', '#', '.', '#', '.']
['.', '.', '.', '.', '.', '.', '#', '.']
['.', '#', '#', '#', '#', '.', '#', '.']
['.', '.', '.', '.', '#', '.', '.', '.']
['#', '#', '#', '.', '#', '#', '#', '.']
['.', '.', '.', '.', '.', '.', '.', '.']
['.', '#', '#', '#', '#', '#', '#', 'G']


In [9]:
# Step 3: Find the start and goal positions

start = None
goal = None

for i in range(rows):
    for j in range(cols):
        if maze[i][j] == 'S':
            start = (i, j)
        elif maze[i][j] == 'G':
            goal = (i, j)

print("Start position:", start)
print("Goal position:", goal)

Start position: (0, 0)
Goal position: (7, 7)


In [10]:
# Step 4: Define possible movements

# The algorithm can move up, down, left, and right
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def get_neighbors(position):
    # This function finds all valid next moves from the current position
    
    x, y = position
    neighbors = []

    for dx, dy in directions:
        nx = x + dx
        ny = y + dy

        # Make sure the move stays inside the maze
        if 0 <= nx < rows and 0 <= ny < cols:
            
            # Make sure the move does not go into a wall
            if maze[nx][ny] != '#':
                neighbors.append((nx, ny))

    return neighbors

In [11]:
# Step 5: Function to display the maze with the final path

def display_maze_with_path(path):
    # Make a copy of the maze so we do not change the original maze
    display_grid = [row.copy() for row in maze]

    # Mark the path using *
    if path:
        for x, y in path:
            if display_grid[x][y] not in ['S', 'G']:
                display_grid[x][y] = '*'

    # Print the maze nicely
    for row in display_grid:
        print(" ".join(row))

In [15]:
# Step 6: Breadth-First Search (BFS)

def bfs(start, goal):
    # BFS explores the maze level by level
    # It uses a queue and usually finds the shortest path
    
    start_time = time.perf_counter()

    queue = deque()
    queue.append((start, [start]))

    visited = set()
    explored_nodes = 0

    while queue:
        current, path = queue.popleft()

        if current in visited:
            continue

        visited.add(current)
        explored_nodes += 1

        if current == goal:
            end_time = time.perf_counter()
            return {
                "algorithm": "BFS",
                "path": path,
                "path_length": len(path),
                "explored_nodes": explored_nodes,
                "runtime": end_time - start_time
            }

        for neighbor in get_neighbors(current):
            if neighbor not in visited:
                queue.append((neighbor, path + [neighbor]))

    return None

In [16]:
# Step 7: Depth-First Search (DFS)

def dfs(start, goal):
    # DFS explores one path deeply first before backtracking
    
    start_time = time.perf_counter()

    stack = []
    stack.append((start, [start]))

    visited = set()
    explored_nodes = 0

    while stack:
        current, path = stack.pop()

        if current in visited:
            continue

        visited.add(current)
        explored_nodes += 1

        if current == goal:
            end_time = time.perf_counter()
            return {
                "algorithm": "DFS",
                "path": path,
                "path_length": len(path),
                "explored_nodes": explored_nodes,
                "runtime": end_time - start_time
            }

        # Reversed order helps show that DFS depends on exploration order
        for neighbor in reversed(get_neighbors(current)):
            if neighbor not in visited:
                stack.append((neighbor, path + [neighbor]))

    return None

In [17]:
# Step 8: Manhattan distance heuristic for A*

def manhattan_distance(position, goal):
    # Manhattan distance estimates how far a position is from the goal
    # It works well for grid mazes where movement is up, down, left, and right
    
    x1, y1 = position
    x2, y2 = goal

    return abs(x1 - x2) + abs(y1 - y2)

In [18]:
# Step 9: A* Search Algorithm

def astar(start, goal):
    # A* uses both the real cost so far and an estimated cost to the goal
    # f(n) = g(n) + h(n)
    # g(n) = cost from start to current position
    # h(n) = heuristic estimate from current position to goal
    
    start_time = time.perf_counter()

    priority_queue = []
    heapq.heappush(priority_queue, (0, 0, start, [start]))

    visited = set()
    explored_nodes = 0

    while priority_queue:
        f_score, g_score, current, path = heapq.heappop(priority_queue)

        if current in visited:
            continue

        visited.add(current)
        explored_nodes += 1

        if current == goal:
            end_time = time.perf_counter()
            return {
                "algorithm": "A*",
                "path": path,
                "path_length": len(path),
                "explored_nodes": explored_nodes,
                "runtime": end_time - start_time
            }

        for neighbor in get_neighbors(current):
            if neighbor not in visited:
                new_g_score = g_score + 1
                h_score = manhattan_distance(neighbor, goal)
                new_f_score = new_g_score + h_score

                heapq.heappush(
                    priority_queue,
                    (new_f_score, new_g_score, neighbor, path + [neighbor])
                )

    return None

In [19]:
# Step 10: Run BFS, DFS, and A*

bfs_result = bfs(start, goal)
dfs_result = dfs(start, goal)
astar_result = astar(start, goal)

results = [bfs_result, dfs_result, astar_result]

for result in results:
    print(result["algorithm"])
    print("Path:", result["path"])
    print("Path length:", result["path_length"])
    print("Explored nodes:", result["explored_nodes"])
    print("Runtime:", round(result["runtime"], 8), "seconds")
    print("-" * 50)

NameError: name 'time' is not defined

In [17]:
# Step 11: Show each path visually

for result in results:
    print(result["algorithm"], "Path Visualization")
    display_maze_with_path(result["path"])
    print("-" * 50)

BFS Path Visualization
S * * * # . . .
# # # * # . # .
. . . * * * # .
. # # # # * # .
. . . . # * * *
# # # . # # # *
. . . . . . . *
. # # # # # # G
--------------------------------------------------
DFS Path Visualization
S * * * # . . .
# # # * # . # .
* * * * . . # .
* # # # # . # .
* * * * # . . .
# # # * # # # .
. . . * * * * *
. # # # # # # G
--------------------------------------------------
A* Path Visualization
S * * * # . . .
# # # * # . # .
. . . * * * # .
. # # # # * # .
. . . . # * * *
# # # . # # # *
. . . . . . . *
. # # # # # # G
--------------------------------------------------


In [ ]:
# Step 12: Final comparison table

print(f"{'Algorithm':<12} {'Path Length':<15} {'Explored Nodes':<17} {'Runtime (sec)':<15}")
print("-" * 65)

for result in results:
    print(f"{result['algorithm']:<12} {result['path_length']:<15} {result['explored_nodes']:<17} {result['runtime']:<15.8f}")

In [ ]:
## Analysis

The results show that BFS and A* both found a shortest path in this maze. DFS also found a valid path, but its path may be longer because DFS explores deeply first and depends on the exploration order.

A* uses the Manhattan distance heuristic to estimate how close each position is to the goal. This makes A* more informed than BFS and DFS because it uses both the actual path cost and an estimate of the remaining distance.

I compared the algorithms using path length, number of explored nodes, and runtime. These metrics help show the strengths and weaknesses of each algorithm.

In [ ]:
## Connection to Research

While working on this project, I explored research related to pathfinding algorithms. I learned that basic methods like BFS and DFS are fundamental search techniques, but more advanced algorithms like A* improve performance by using heuristics.

In particular, I looked at the A* algorithm introduced by Hart, Nilsson, and Raphael (1968). This algorithm combines the actual cost of a path with an estimated distance to the goal, which helps guide the search more efficiently.

After understanding this, I was able to better explain my own results. I noticed that A* performed better in my project because it makes more informed decisions compared to BFS and DFS.

This helped me connect what I implemented in my code to how these algorithms are used in real-world applications like navigation and robotics.

In [ ]:
## References

Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A Formal Basis for the Heuristic Determination of Minimum Cost Paths. IEEE Transactions on Systems Science and Cybernetics.

Russell, S., & Norvig, P. (2021). Artificial Intelligence: A Modern Approach.